# Week 1 – Data Cleaning & Preprocessing
## Titanic Dataset

**Objective:** Acquire, inspect, clean and preprocess a publicly available Titanic dataset using Python so it is ready for further analysis and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## 1. Load the Dataset

Download `train.csv` from the Kaggle Titanic competition and place it in the same folder as this notebook.

In [ ]:
df = pd.read_csv('train.csv')
print("Dataset shape:", df.shape)
display(df.head())

## 2. Initial Data Exploration

The first inspection checks the structure, data types, descriptive statistics, and missing values before cleaning.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nSummary statistics:")
display(df.describe(include='all').T)

In [ ]:
print("Missing values:")
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
display(pd.DataFrame({"Missing Values": missing, "Percentage": missing_pct})[
    lambda x: x["Missing Values"] > 0
])

## 3. Missing-Value Handling

For this academic preprocessing workflow, missing `Age` values are filled with the median because age is numerical and the median is less sensitive to extreme values. Missing `Embarked` values are filled with the mode. `Cabin` has substantial missingness, so a separate `Unknown` category is used rather than deleting all rows.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Cabin'] = df['Cabin'].fillna('Unknown')

print("Remaining missing values:")
display(df.isnull().sum()[df.isnull().sum() > 0])

## 4. Duplicate Records

In [ ]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after cleaning:", df.duplicated().sum())

## 5. Erroneous and Consistency Checks

The dataset is checked for impossible or suspicious values such as invalid ages, fares, passenger classes, and family counts.

In [ ]:
checks = {
    'Age below 0': (df['Age'] < 0).sum(),
    'Fare below 0': (df['Fare'] < 0).sum(),
    'Pclass outside 1–3': (~df['Pclass'].isin([1,2,3])).sum(),
    'SibSp below 0': (df['SibSp'] < 0).sum(),
    'Parch below 0': (df['Parch'] < 0).sum()
}
display(pd.Series(checks, name='Invalid records'))

## 6. Outlier Detection – Fare

The IQR method is used as an initial screening technique. Outliers are not automatically deleted because an extreme fare can represent a genuine passenger observation.

In [ ]:
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

fare_outliers = df[(df['Fare'] < lower) | (df['Fare'] > upper)]
print("Fare IQR lower bound:", lower)
print("Fare IQR upper bound:", upper)
print("Number of Fare outliers:", len(fare_outliers))

## 7. Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['Age'], bins=20)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Number of Passengers')
plt.tight_layout()
plt.show()

## 8. Survival Distribution

In [ ]:
survival_counts = df['Survived'].value_counts().sort_index()
survival_counts.index = ['Did not survive (0)', 'Survived (1)']

plt.figure(figsize=(7,5))
survival_counts.plot(kind='bar')
plt.title('Survival Distribution')
plt.xlabel('Outcome')
plt.ylabel('Number of Passengers')
plt.tight_layout()
plt.show()

## 9. Fare Outlier Visualization

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(df['Fare'].dropna())
plt.title('Fare Outlier Detection')
plt.ylabel('Fare')
plt.tight_layout()
plt.show()

## 10. Feature Engineering

A simple `FamilySize` feature is created from `SibSp` and `Parch`. A binary `IsAlone` feature indicates whether the passenger travelled without other recorded family members.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

display(df[['SibSp','Parch','FamilySize','IsAlone']].head())

## 11. Categorical Encoding

Categorical features can be converted into numerical columns using one-hot encoding. The target variable `Survived` is kept separately.

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_encoded = pd.get_dummies(X, columns=['Sex','Embarked'], drop_first=True)
print("Encoded feature shape:", X_encoded.shape)
display(X_encoded.head())

## 12. Final Quality Check

In [ ]:
print("Final shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("Total duplicate rows:", df.duplicated().sum())
display(df.head())

## 13. Save the Cleaned Dataset

In [ ]:
df.to_csv('titanic_cleaned.csv', index=False)
print("Saved titanic_cleaned.csv")

## 14. Potential Impact of Preprocessing

- Missing-value treatment prevents incomplete records from causing problems during analysis.
- Removing exact duplicates prevents repeated observations from receiving extra influence.
- Outlier screening identifies unusual fares without automatically deleting legitimate observations.
- Consistency checks help identify impossible values.
- Feature engineering creates potentially useful variables for later modeling.
- One-hot encoding converts categorical variables into machine-readable features.
- The cleaned dataset is easier to use for exploratory analysis and machine-learning models.

## Conclusion

The Titanic dataset provides a useful example of data acquisition, exploratory analysis, missing-value handling, duplicate checking, outlier screening, feature engineering and categorical preprocessing. Every transformation should be documented so that the workflow remains reproducible.